# CubeMind two-stage training (RunPod RTX 5090 — 32GB)

Mid-scale validation between Colab Blackwell (d=384, ~33M params, ~10 min) and the
H200 production run (d=768, ~200M params, ~6h, $25-50). The 5090's 32GB lets you run
**the exact H200 architecture** at reduced batch+seq so loss trajectories transfer cleanly.

**Cost expectation:** RunPod 5090 community is ~$0.30-0.50/hr. Stage 1 (10K steps) ≈
1.5-2.5h, stage 2 (3K steps) ≈ 30-45min. Total ~$0.80-1.50 per full run.

**Pipeline mirrors `run_h200.sh`:**
1. Pre-flight: tokenizer, LM corpus, multitask label-range check
2. Stage 1: LM pretrain on c4 + OpenThoughts, full hybrid stack on
3. Stage 2: load stage-1 best, freeze backbone, train 5 multitask heads

If both stages converge cleanly here, the same script + same flags run at H200 scale.

## 1. Bootstrap the pod

Run `scripts/runpod_h200_bootstrap.sh` from a fresh RunPod terminal first (it works
for any NVIDIA pod, not just H200). That installs uv + torch + tokenizers and clones
the repo to `/workspace/cubemind`. This notebook assumes you're running it from inside
`/workspace/cubemind` after bootstrap completes.

In [ ]:
import os, sys, subprocess, shutil, time, json
from pathlib import Path

WORKSPACE = Path('/workspace')
REPO      = WORKSPACE / 'cubemind'
DATA      = WORKSPACE / 'data'
TOKENIZER = WORKSPACE / 'tokenizer'
RESULTS   = WORKSPACE / 'results_5090'
for p in (DATA, TOKENIZER, RESULTS): p.mkdir(parents=True, exist_ok=True)

os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', 'dev'], check=True)
subprocess.run(['git', 'pull', 'origin', 'dev'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)

## 2. Upload data (run once per pod)

Use `scp` from your laptop to push the four required files to `/workspace`. The notebook
checks for them next:

```bash
scp grillcheese_spm32k_v2.model           root@<pod-ip>:/workspace/tokenizer/
scp allenai_c4_realnewslike.500m_tokens.txt  root@<pod-ip>:/workspace/data/
scp openthoughts_chat.txt                 root@<pod-ip>:/workspace/data/
scp multitask_combined_v3_clean.jsonl     root@<pod-ip>:/workspace/data/multitask_combined.jsonl
```

Sizes: tokenizer ~1MB, c4 ~2GB, OpenThoughts ~2.5GB, multitask ~300MB. ~5GB total.

In [ ]:
TOKENIZER_PATH = TOKENIZER / 'grillcheese_spm32k_v2.model'
C4_PATH        = DATA / 'allenai_c4_realnewslike.500m_tokens.txt'
OT_PATH        = DATA / 'openthoughts_chat.txt'
MT_PATH        = DATA / 'multitask_combined.jsonl'  # must be the v3_clean file
STAGE1_LM      = DATA / 'stage1_lm_combined.txt'

for p, label in [(TOKENIZER_PATH, 'tokenizer'),
                 (C4_PATH, 'c4 LM corpus'),
                 (OT_PATH, 'OpenThoughts chat'),
                 (MT_PATH, 'multitask v3_clean')]:
    if p.exists():
        print(f'  ok  {label:<22} {p.stat().st_size/1e6:>9.1f} MB  {p}')
    else:
        print(f'  MISSING  {label:<22}                {p}')

## 3. Build the stage-1 combined corpus

On 5090 we use the **full** c4 + OpenThoughts (vs Colab's 50MB c4 slice) — a ~5GB combined
corpus is easily indexable on the local NVMe. Done once per pod.

In [ ]:
if not STAGE1_LM.exists():
    print('concatenating c4 + OpenThoughts...')
    with STAGE1_LM.open('wb') as out:
        for src in (C4_PATH, OT_PATH):
            with src.open('rb') as f:
                shutil.copyfileobj(f, out, length=64*1024*1024)
print(f'  stage1 corpus: {STAGE1_LM} ({STAGE1_LM.stat().st_size/1e9:.2f} GB)')

## 4. Pre-flight — multitask label-range check

Same check `run_h200.sh` does. Catches out-of-range labels BEFORE the 15-min compile pass.
If this fails, run `scrub_multitask.py` first (the v3_clean file should pass).

In [ ]:
import io
limits = {'opcode_id':55,'intent_id':6,'schema_id':16,'rule_id':32,'validity':2}
maxes = {}
with io.open(str(MT_PATH),'r',encoding='utf-8',errors='replace') as f:
    for line in f:
        try: row = json.loads(line)
        except: continue
        for k in limits:
            if k in row: maxes[k] = max(maxes.get(k,0), int(row[k]))
ok = True
for k, lim in limits.items():
    seen = maxes.get(k)
    if seen is None: print(f'  WARN {k}: missing in all rows')
    elif seen >= lim: print(f'  FAIL {k}: max={seen} >= head size {lim}'); ok = False
    else: print(f'  ok   {k}: max={seen} < head size {lim}')
assert ok, 'multitask labels out of range — run scrub_multitask.py'

## 5. Stage 1 — LM pretrain at H200 architecture (~1.5-2.5h on 5090)

**Architecture matches `run_h200.sh` exactly** (d=768, L=12, d_ffn=3072) — the only changes
are batch+seq, scaled to fit 32GB:

| | H200 (run_h200.sh) | RTX 5090 (this notebook) |
|---|---|---|
| seq_len | 1024 | 512 |
| batch_size | 32 | 12 |
| grad_accum | 8 | 8 |
| effective batch | 256 | 96 |
| steps | 20,000 | 10,000 |
| tokens/step | 32,768 | 6,144 |
| total tokens | 5.2B | 614M |

Same LR schedule, same hybrid stack, same dtype — just less data per step.
Use `--compile` to fuse MinGRU prefix scan + RMSNorm + MoE; eats 15-25 min upfront.

In [ ]:
S1_DIR = RESULTS / 'stage1_lm'
if S1_DIR.exists():
    shutil.move(str(S1_DIR), str(S1_DIR.with_name(f'stage1_lm_pre_{int(time.time())}')))
S1_DIR.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env['RESULTS_DIR_OVERRIDE'] = str(S1_DIR)

cmd = ['python', '-u', 'sandbox/mingru_baseline/train_torch.py',
       '--tokenizer-path', str(TOKENIZER_PATH),
       '--d-model', '768', '--n-layers', '12', '--d-ffn', '3072',
       '--vocab', '32128', '--seq-len', '512',
       '--steps', '10000', '--batch-size', '12', '--grad-accum', '8',
       '--lr', '6e-4', '--min-lr', '6e-5', '--warmup', '1000',
       '--wd', '0.01', '--clip', '1.0', '--dtype', 'bf16', '--compile',
       '--log-every', '50', '--eval-every', '500', '--ckpt-every', '1000',
       '--prompts', 'sandbox/mingru_baseline/prompts_news.txt',
       '--data-path', str(STAGE1_LM),
       '--vsa-binding-head', '--vsa-binding-d', '10240',
       '--moe', '--moe-experts', '4', '--moe-top-k', '2',
       '--attention', '--attn-heads', '4', '--attn-window', '128', '--attn-every-n', '3',
       '--memory', '--mem-max', '200', '--mem-write-threshold', '0.4',
       '--mem-every-n', '4', '--mem-consolidate-every', '1000',
       '--hypergrad']
print(' '.join(cmd))
subprocess.run(cmd, env=env, check=True)

In [ ]:
S1_BEST = S1_DIR / 'best.pt'
assert S1_BEST.exists(), 'stage 1 produced no best.pt — check the train log'
S1_SAVED = WORKSPACE / 'stage1_5090_best.pt'
shutil.copy(str(S1_BEST), str(S1_SAVED))
print(f'  stage 1 best: {S1_SAVED} ({S1_SAVED.stat().st_size/1e6:.1f} MB)')

## 6. Stage 2 — multitask head fine-tune on frozen base (~30-45 min)

Loads stage 1 weights, freezes the backbone, trains only the 5 multitask heads + the
binding head's query projection.

**All 5 head loss weights enabled** (the validation run had rule + validity at 0 by default).

In [ ]:
S2_DIR = RESULTS / 'stage2_multitask'
if S2_DIR.exists():
    shutil.move(str(S2_DIR), str(S2_DIR.with_name(f'stage2_multitask_pre_{int(time.time())}')))
S2_DIR.mkdir(parents=True, exist_ok=True)
env['RESULTS_DIR_OVERRIDE'] = str(S2_DIR)

# Stage 2 doesn't benefit from --compile (small frozen model, short run, compile cache
# misses on the new MinGRUMultiTask wrapper anyway — see colab validation).
cmd = ['python', '-u', 'sandbox/mingru_baseline/train_torch.py',
       '--tokenizer-path', str(TOKENIZER_PATH),
       '--d-model', '768', '--n-layers', '12', '--d-ffn', '3072',
       '--vocab', '32128', '--seq-len', '512',
       '--steps', '3000', '--batch-size', '32', '--grad-accum', '2',
       '--lr', '2e-4', '--min-lr', '2e-5', '--warmup', '200',
       '--wd', '0.01', '--clip', '1.0', '--dtype', 'bf16',
       '--log-every', '25', '--eval-every', '200', '--ckpt-every', '500',
       '--use-jsonl-dataset', '--jsonl-path', str(MT_PATH),
       '--aux-opcode-loss-weight',   '0.4',
       '--aux-intent-loss-weight',   '0.2',
       '--aux-schema-loss-weight',   '0.2',
       '--aux-rule-loss-weight',     '0.2',
       '--aux-validity-loss-weight', '0.1',
       '--vsa-binding-head', '--vsa-binding-d', '10240',
       '--moe', '--moe-experts', '4', '--moe-top-k', '2',
       '--attention', '--attn-heads', '4', '--attn-window', '128', '--attn-every-n', '3',
       '--memory', '--mem-max', '200', '--mem-write-threshold', '0.4',
       '--mem-every-n', '4', '--mem-consolidate-every', '1000',
       '--hypergrad',
       '--init-from', str(S1_SAVED), '--freeze-backbone']
print(' '.join(cmd))
subprocess.run(cmd, env=env, check=True)

## 7. Validation summary

What you want to see:

**Stage 1:**
- val CE descending past 2.5 by step 5000, target ~2.0-2.2 by step 10000
- gn bounded 0.4-1.5
- Sleep consolidation firing every 1000 steps
- No torch.compile recompile warnings after step 100

**Stage 2:**
- LM CE roughly flat (frozen backbone)
- All 5 per-head accuracies climbing past their random baselines:
  - opcode > 0.6 (random ~0.02), intent > 0.4 (random ~0.17), schema > 0.6 (random ~0.06)
  - rule > 0.5 (random ~0.03 — most rule mass in 'other' bucket)
  - validity > 0.7 (random 0.5)

If all of the above hold, the H200 production run is safe to launch.  Pull the checkpoints
back with `scp -r root@<pod-ip>:/workspace/results_5090 ./` to keep them off the pod (it's
ephemeral).

In [ ]:
for s_dir, label in [(S1_DIR, 'stage 1'), (S2_DIR, 'stage 2')]:
    summary = s_dir / 'summary.json'
    if not summary.exists():
        print(f'  {label}: no summary.json'); continue
    s = json.loads(summary.read_text())
    print(f'\n{label} ({s_dir.name}):')
    print(f'  params:      {s["params"]:,}')
    print(f'  steps:       {s["steps"]:,}')
    print(f'  tokens seen: {s["tokens_seen"]:,}')
    print(f'  elapsed:     {s["elapsed_min"]:.1f} min')
    print(f'  tok/s:       {s["tokens_per_sec"]:,.0f}')
    print(f'  final val CE: {s["final_val_ce"]:.4f}  (PPL {s["final_val_ppl"]:.2f})')
    print(f'  best  val CE: {s["best_val_ce"]:.4f}  (PPL {s["best_val_ppl"]:.2f})')